# 08 - Labeling Sentimen (2 Annotator) + Inter-Rater Agreement

Tahap 3.1.4.2 (Pelabelan Ground Truth) pada proposal TA. Sample gold-standard dari tahap 07
dilabel manual oleh 2 annotator independen (Shafwa & Khalila), 3 kelas (negatif/netral/positif),
lalu dicek konsistensinya (Cohen's Kappa) dan baris yang tidak sepakat diresolusi manual.

**Alur:**
1. `predict_model_labels()` -- jalankan model pre-trained IndoBERTweet sentiment di sample, HANYA
   untuk ditampilkan ke annotator SETELAH mereka memilih label sendiri (bukan sebelum -- supaya
   tidak bias). Otomatis/tidak interaktif.
2. **[MANUAL, di luar notebook ini]** -- 2 annotator melabel independen via web app ->
   `labels_shafwa.csv`, `labels_khalila.csv`.
3. `compute_inter_rater_agreement()` -- Cohen's Kappa (3-class & binary), confusion matrix,
   per-class agreement -> `inter_rater_merged.csv` + baris yang tidak sepakat untuk direview.
4. **[MANUAL, di luar notebook ini]** -- baris yang tidak sepakat diputuskan satu per satu ->
   `disagreements_label_final.csv`.

**Konsolidasi dari:**
- `02_IndoBert/01_praprocessing/prediksi.py` -> `predict_model_labels()`
- `02_IndoBert/01_praprocessing/inter_rater_kappa.ipynb` (bagian hitung & export) ->
  `compute_inter_rater_agreement()`

**Sengaja tidak diikutkan:**
- `labeling.py`/`labeling_non_negatif.py` (Flask app, interaktif) -- labeling SUDAH SELESAI
  (2.137 baris per annotator). Untuk label data baru, jalankan file aslinya langsung.
- `konversi_label.py` -- skema binary diturunkan langsung dari label 3-class saat dibutuhkan,
  bukan dari konversi file terpisah (lihat `compute_inter_rater_agreement()` dan tahap 09).

**Catatan eksekusi:** `predict_model_labels()` butuh model ~500MB diunduh + inferensi (beberapa
menit di CPU). Skip otomatis kalau output sudah ada. Resolusi disagreement (langkah 4) murni
keputusan manusia per-komentar, tidak ada logic otomatis yang menggantikannya.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# -------------------------------------------------------------------------
# KONFIGURASI PATH
# -------------------------------------------------------------------------
FINAL_OUTPUT_ROOT = Path.cwd() / "output"
OUTPUT_DIR = FINAL_OUTPUT_ROOT / "08_labeling_sentimen"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPEL_LABELING = FINAL_OUTPUT_ROOT / "07_preprocessing_komentar" / "sampel_labeling.csv"

PREDIKSI_MODEL = OUTPUT_DIR / "prediksi_model.csv"
LABELS_SHAFWA = OUTPUT_DIR / "labels_shafwa.csv"     # hasil labeling manual (IRREPLACEABLE)
LABELS_KHALILA = OUTPUT_DIR / "labels_khalila.csv"   # hasil labeling manual (IRREPLACEABLE)
INTER_RATER_MERGED = OUTPUT_DIR / "inter_rater_merged.csv"
INTER_RATER_DISAGREEMENTS = OUTPUT_DIR / "inter_rater_disagreements.csv"
DISAGREEMENTS_LABEL_FINAL = OUTPUT_DIR / "disagreements_label_final.csv"  # resolusi manual (IRREPLACEABLE)

MODEL_NAME = "Aardiiiiy/indobertweet-base-Indonesian-sentiment-analysis"
NAMA_LABEL = {
    "negative": "negatif", "neutral": "netral", "positive": "positif",
    "negatif": "negatif", "netral": "netral", "positif": "positif",
}
print("Config siap.")

## STEP 1 - Prediksi model (ditampilkan ke annotator SETELAH mereka memilih)

In [ ]:
def predict_model_labels(batch_size: int = 32, max_len: int = 128,
                          output_csv: Path = PREDIKSI_MODEL) -> pd.DataFrame:
    if output_csv.exists():
        print(f"[Prediksi Model] Sudah ada -> {output_csv} (skip)")
        return pd.read_csv(output_csv)

    if not SAMPEL_LABELING.exists():
        raise FileNotFoundError(f"{SAMPEL_LABELING} tidak ada. Jalankan tahap 07 dulu.")

    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    df = pd.read_csv(SAMPEL_LABELING, dtype=str, keep_default_na=False, encoding="utf-8")
    texts = df["comment_text_clean"].tolist()
    print(f"[Prediksi Model] {len(texts):,} komentar | model: {MODEL_NAME}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)

    id2label_raw = model.config.id2label
    id2nama = {}
    for k, v in id2label_raw.items():
        nama = NAMA_LABEL.get(str(v).strip().lower())
        if nama is None:
            raise ValueError(f"id2label model tidak dikenali: {id2label_raw}")
        id2nama[int(k)] = nama
    if sorted(id2nama.values()) != ["negatif", "netral", "positif"]:
        raise ValueError(f"Label model harus tepat 3 kelas, dapat: {id2nama}")

    probs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=max_len,
                             return_tensors="pt").to(device)
            batch_probs = torch.softmax(model(**enc).logits, dim=-1).cpu().tolist()
            probs.extend(batch_probs)
            print(f"   {min(i + batch_size, len(texts)):,}/{len(texts):,}", end="\r")
    print()

    out = pd.DataFrame({"row_id": range(len(df)), "comment_text_clean": texts})
    for idx, nama in id2nama.items():
        out[f"p_{nama}"] = [round(p[idx], 4) for p in probs]
    kolom_p = [f"p_{n}" for n in ("negatif", "netral", "positif")]
    out["model_label"] = out[kolom_p].idxmax(axis=1).str.replace("p_", "", regex=False)
    out["model_conf"] = out[kolom_p].max(axis=1)

    out.to_csv(output_csv, index=False, encoding="utf-8")
    print(f"[Prediksi Model] Tersimpan -> {output_csv}")
    print(out["model_label"].value_counts().to_string())
    return out


predict_model_labels()

## [MANUAL] 2 annotator melabel independen via web app
Jalankan `labeling.py` (di luar notebook ini), hasil tersimpan ke `labels_shafwa.csv` &
`labels_khalila.csv`. Lanjut ke cell berikutnya setelah labeling selesai.

## STEP 2 - Hitung Cohen's Kappa + export agreement/disagreement

In [ ]:
def compute_inter_rater_agreement() -> dict:
    """Return dict berisi df merged, kappa 3-class & binary, siap dipakai cell evaluasi di bawah."""
    if not LABELS_SHAFWA.exists() or not LABELS_KHALILA.exists():
        raise FileNotFoundError(
            f"{LABELS_SHAFWA} / {LABELS_KHALILA} tidak ada. Perlu hasil labeling manual 2 annotator."
        )

    df_s = pd.read_csv(LABELS_SHAFWA).rename(columns={"label_manual": "label_shafwa"})
    df_k = pd.read_csv(LABELS_KHALILA).rename(columns={"label_manual": "label_khalila"})

    if "comment_text_clean" in df_s.columns and "comment_text_clean" in df_k.columns:
        df = pd.merge(df_s[["comment_text_clean", "label_shafwa"]],
                       df_k[["comment_text_clean", "label_khalila"]],
                       on="comment_text_clean", how="inner")
    else:
        df = pd.DataFrame({
            "comment_text_clean": df_s["comment_text_clean"].values,
            "label_shafwa": df_s["label_shafwa"].values,
            "label_khalila": df_k["label_khalila"].values,
        })

    df = df.dropna(subset=["label_shafwa", "label_khalila"])
    df["label_shafwa"] = df["label_shafwa"].str.lower().str.strip()
    df["label_khalila"] = df["label_khalila"].str.lower().str.strip()

    kappa_3class = cohen_kappa_score(df["label_shafwa"], df["label_khalila"])

    df["binary_shafwa"] = df["label_shafwa"].apply(lambda x: "negatif" if x == "negatif" else "non-negatif")
    df["binary_khalila"] = df["label_khalila"].apply(lambda x: "negatif" if x == "negatif" else "non-negatif")
    kappa_binary = cohen_kappa_score(df["binary_shafwa"], df["binary_khalila"])

    df["agree_3class"] = df["label_shafwa"] == df["label_khalila"]
    df["agree_binary"] = df["binary_shafwa"] == df["binary_khalila"]
    df["disagreement_type"] = np.where(
        df["agree_3class"], "AGREE", df["label_shafwa"] + " -> " + df["label_khalila"]
    )

    if not INTER_RATER_MERGED.exists():
        df.to_csv(INTER_RATER_MERGED, index=False)
        print(f"[Kappa] Tersimpan -> {INTER_RATER_MERGED}")
    else:
        print(f"[Kappa] Sudah ada -> {INTER_RATER_MERGED} (skip tulis ulang)")

    df_disagree = df[~df["agree_3class"]][["comment_text_clean", "label_shafwa", "label_khalila", "disagreement_type"]]
    if not INTER_RATER_DISAGREEMENTS.exists():
        df_disagree.to_csv(INTER_RATER_DISAGREEMENTS, index=False)
        print(f"[Kappa] Tersimpan -> {INTER_RATER_DISAGREEMENTS}")
    else:
        print(f"[Kappa] Sudah ada -> {INTER_RATER_DISAGREEMENTS} (skip tulis ulang)")

    overall_agree = df["agree_3class"].mean() * 100
    print(f"[Kappa] Cohen's Kappa 3-class: {kappa_3class:.4f} | binary: {kappa_binary:.4f} | "
          f"Overall agreement: {overall_agree:.1f}% | Disagreement: {len(df_disagree)} baris")

    if not DISAGREEMENTS_LABEL_FINAL.exists():
        print(f"[Kappa] {DISAGREEMENTS_LABEL_FINAL.name} belum ada -- "
              f"{len(df_disagree)} baris disagreement perlu diresolusi manual satu per satu "
              f"sebelum lanjut ke tahap 09 (kolom: label_final, keputusan).")

    return {"df": df, "kappa_3class": kappa_3class, "kappa_binary": kappa_binary,
            "df_disagree": df_disagree}


hasil_kappa = compute_inter_rater_agreement()

## [MANUAL] Resolusi disagreement
Baris yang tidak sepakat diputuskan satu per satu -> `disagreements_label_final.csv`
(kolom `label_final`, `keputusan`). Lanjut ke cell berikutnya setelah resolusi selesai (boleh parsial).

## Stats/Grafik -- Inter-Rater Reliability (Cohen's Kappa)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score, confusion_matrix

df = pd.read_csv(OUTPUT_DIR / "inter_rater_merged.csv")
n = len(df)

kappa_3class = cohen_kappa_score(df["label_shafwa"], df["label_khalila"])
kappa_binary = cohen_kappa_score(df["binary_shafwa"], df["binary_khalila"])


def interpret_kappa(k):
    if k < 0: return "Buruk (worse than chance)"
    elif k < 0.20: return "Sangat Lemah"
    elif k < 0.40: return "Lemah"
    elif k < 0.60: return "Sedang (Moderate)"
    elif k < 0.80: return "Baik (Substantial)"
    else: return "Sangat Baik (Almost Perfect)"


print(f"Total baris        : {n}")
print(f"Kappa 3-class       : {kappa_3class:.4f} -> {interpret_kappa(kappa_3class)}")
print(f"Kappa binary        : {kappa_binary:.4f} -> {interpret_kappa(kappa_binary)}")
print(f"Overall agreement   : {df['agree_3class'].mean()*100:.1f}%")

### 1. Distribusi Label per Annotator

In [ ]:
dist_df = pd.DataFrame({
    "Shafwa": df["label_shafwa"].value_counts(),
    "Khalila": df["label_khalila"].value_counts(),
}).fillna(0).astype(int)
dist_df["Shafwa_%"] = (dist_df["Shafwa"] / n * 100).round(1)
dist_df["Khalila_%"] = (dist_df["Khalila"] / n * 100).round(1)
print(dist_df)

colors = {"negatif": "#e74c3c", "netral": "#95a5a6", "positif": "#2ecc71"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (labeler, col) in zip(axes, [("Shafwa", "label_shafwa"), ("Khalila", "label_khalila")]):
    counts = df[col].value_counts()
    counts.plot(kind="bar", ax=ax, color=[colors.get(l, "#3498db") for l in counts.index], edgecolor="white")
    ax.set_title(f"Distribusi Label - {labeler}")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

### 2. Confusion Matrix Antar Annotator

In [ ]:
labels_all = sorted(set(df["label_shafwa"]) | set(df["label_khalila"]))
cm = confusion_matrix(df["label_shafwa"], df["label_khalila"], labels=labels_all)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_all, yticklabels=labels_all, ax=axes[0])
axes[0].set_xlabel("Label Khalila"); axes[0].set_ylabel("Label Shafwa")
axes[0].set_title(f"3-class (kappa={kappa_3class:.3f})")

cm_bin = confusion_matrix(df["binary_shafwa"], df["binary_khalila"], labels=["negatif", "non-negatif"])
sns.heatmap(cm_bin, annot=True, fmt="d", cmap="Reds", xticklabels=["negatif","non-negatif"],
            yticklabels=["negatif","non-negatif"], ax=axes[1])
axes[1].set_xlabel("Label Khalila"); axes[1].set_ylabel("Label Shafwa")
axes[1].set_title(f"Binary (kappa={kappa_binary:.3f})")
plt.tight_layout()
plt.show()

### 3. Per-Class Agreement

In [ ]:
for label in sorted(df["label_shafwa"].unique()):
    mask_s = df["label_shafwa"] == label
    mask_k = df["label_khalila"] == label
    both = (mask_s & mask_k).sum()
    only_s = (mask_s & ~mask_k).sum()
    only_k = (~mask_s & mask_k).sum()
    total_any = (mask_s | mask_k).sum()
    print(f"[{label.upper()}] keduanya setuju={both} | hanya Shafwa={only_s} | hanya Khalila={only_k} | "
          f"agreement={both/total_any*100 if total_any else 0:.1f}%")

### 4. Disagreement: Tipe & Contoh Komentar Paling Ambigu

In [ ]:
df_disagree = df[~df["agree_3class"]].copy()
print(f"Total disagreement: {len(df_disagree)} ({len(df_disagree)/n*100:.1f}%)")

counts = df_disagree["disagreement_type"].value_counts()
fig, ax = plt.subplots(figsize=(9, 4))
counts.plot(kind="barh", ax=ax, color="#e67e22", edgecolor="white")
ax.set_title("Tipe Disagreement (Shafwa -> Khalila)")
plt.tight_layout()
plt.show()

pd.set_option("display.max_colwidth", 120)
print("\nContoh ambigu negatif <-> netral:")
print(df_disagree[df_disagree["disagreement_type"].isin(["negatif -> netral", "netral -> negatif"])]
        [["comment_text_clean", "label_shafwa", "label_khalila"]].head(10))

### 5. Status Resolusi Manual Disagreement

In [ ]:
final_path = OUTPUT_DIR / "disagreements_label_final.csv"
if final_path.exists():
    df_final = pd.read_csv(final_path)
    print(f"Disagreement sudah diresolusi: {len(df_final)}/{len(df_disagree)}")
    print(df_final["keputusan"].value_counts())
else:
    print("disagreements_label_final.csv belum ada -- resolusi manual belum dimulai/selesai.")